# 30 — Compose QA review gate → `v6p5_compose_<hash>`
**No auto-promotion.** Every TRAIN candidate needs `KEEP` or `DROP` in
`artifacts/qa/v6p5_review_decisions.csv` (blank = pending). `REVIEW_THEN_KEEP` is a
recommendation, never a decision. The stage stops with `needs_review` until all TRAIN
candidates are resolved. Exact SHA duplicates are a hard gate; pHash candidates must be reviewed.

Workflow: run → open the decisions CSV (spreadsheet) → fill decision/reviewer/comment → run again.

In [ ]:
# Bootstrap: make the project package importable from notebooks/.
import os, sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("YOLOV11SDI_PROJECT_ROOT", str(ROOT))

import pandas as pd
from IPython.display import display, Image as IPImage
from yolov11sdi import pipeline
from yolov11sdi.config import load_config
from yolov11sdi.paths import ProjectPaths
from yolov11sdi.state import StateStore

FORCE_REBUILD = False   # True = rebuild even if a matching complete result exists
DRY_RUN = False
paths = ProjectPaths.detect().ensure()
store = StateStore(paths.state)
print("env:", paths.env.value, "| home:", paths.home, "| data:", paths.data_root)

In [ ]:
outcome = pipeline.run_stage("compose", force=FORCE_REBUILD, dry_run=DRY_RUN)
print(outcome)
display(pd.Series(outcome.summary, dtype=object).to_frame("value"))

In [ ]:
cfg = load_config("v6p5_compose")
dec_path = paths.resolve(cfg.get("compose.decisions_file"))
dec = pd.read_csv(dec_path, keep_default_na=False) if dec_path.exists() else pd.DataFrame()
print("decisions file:", dec_path)
if len(dec):
    display(dec.assign(decision=dec["decision"].replace("", "PENDING"))
            .groupby(["candidate_type", "source", "target_split", "decision"]).size().rename("n").reset_index())

In [ ]:
# Pending TRAIN candidates + their contact sheets (stage QA extract).
pending = paths.compose / "pending_review.csv"
if pending.exists():
    pend = pd.read_csv(pending, keep_default_na=False); display(pend.head(50))
ph = paths.compose / "phash_candidates_to_review.csv"
if ph.exists():
    print("pHash candidates to review:"); display(pd.read_csv(ph))
extract = sorted((paths.working / "compose_qa_extract").glob("*/contacts/*.jpg"))
for p in extract[:load_config("v6p5_compose").get("compose.max_contact_sheets")]:
    print(p.name); display(IPImage(filename=str(p), width=640))

### Review gate
If the status above is `needs_review`, **stop here**, fill the decisions CSV, then re-run this notebook.
After a `complete` compose, notebook 40 auto-selects the compose tag.